# Fish2 ROI explorer — fresh soma-in-ROI query + previous morphology cluster examples

This notebook is designed for students and Google Colab.

## What it does

1. Connects to the Fish2 neuPrint dataset.
2. Lists all ROI names.
3. Lets the user change `ROI_QUERY` (default: `Raphe_Superior`).
4. **Freshly queries neuPrint every run** for all neurons touching the selected ROI(s).
5. Retrieves soma/root coordinates from neuPrint.
6. Filters to cells whose soma is in the selected ROI region:
   - first using an explicit soma-ROI metadata column, if Fish2 exposes one;
   - otherwise using the same live synapse/contact-cloud spatial proxy used in the local analysis workflow.
7. Saves the fresh results as CSV files and prints the complete body-ID list.
8. Plots the selected soma positions in 3D.
9. Plots selected skeletons.
10. Separately loads the **previous morphology clustering outputs** so students can inspect the previously identified **cluster 7**, its IDs, skeletons, and UMAP location.

> The fresh ROI query and the previous morphology clustering are intentionally separate. Changing `ROI_QUERY` changes the fresh neuPrint population, but does not rewrite the historical cluster assignments.

## Important distinction

A neuron can touch an ROI with an axon or dendrite while its soma is elsewhere.

Therefore the notebook performs two stages:

```text
all neurons touching selected ROI(s)
            ↓
keep neurons whose soma/root coordinate lies in the selected ROI region
```

The resulting fresh soma-in-ROI population is exported every run.

In [ ]:
# ============================================================
# 0. Install dependencies
# ============================================================
import sys
import subprocess
import importlib.util

required = {
    "neuprint": "neuprint-python",
    "pandas": "pandas",
    "numpy": "numpy",
    "plotly": "plotly",
    "tqdm": "tqdm",
    "scipy": "scipy",
}

missing = [
    pip_name
    for module_name, pip_name in required.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

print("Dependencies ready.")

In [ ]:
# ============================================================
# 1. Colab bootstrap: clone the GitHub repository
# ============================================================
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/mduram/fish2-raphe-morphology.git"
REPO_DIR = Path("/content/fish2-raphe-morphology")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

In [ ]:
# ============================================================
# 2. Imports and configuration
# ============================================================
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from scipy.spatial import cKDTree

from neuprint import (
    Client,
    fetch_neurons,
    fetch_synapses,
    NeuronCriteria as NC,
    SynapseCriteria as SC,
)

from tqdm.auto import tqdm
from IPython.display import display

# ----------------------------
# neuPrint
# ----------------------------
NEUPRINT_SERVER = "neuprint-fish2.janelia.org"
NEUPRINT_DATASET = "fish2"

# Paste the token manually before running.
# WARNING: restore the placeholder before pushing to a public repository.
NEUPRINT_TOKEN = "PASTE_NEUPRINT_TOKEN_HERE"

# ----------------------------
# Fresh ROI query
# ----------------------------
# Students can change this.
ROI_QUERY = "Raphe_Superior"

# Match every ROI name containing ROI_QUERY, case-insensitive.
ROI_MATCH_MODE = "contains"

# Optional hard cap while testing. Use None for the full fresh query.
MAX_TOUCHING_NEURONS = None

# ----------------------------
# Fresh soma-in-ROI proxy parameters
# ----------------------------
# These mirror the local workflow concept:
# robust percentile box + PCA ellipsoid + nearest-contact constraint.

SOMA_PROXY_PERCENTILE_LOW = 1.0
SOMA_PROXY_PERCENTILE_HIGH = 99.0

# Coordinate-unit padding around the percentile box.
SOMA_PROXY_PADDING = 1000.0

SOMA_PROXY_USE_ELLIPSOID = True
SOMA_PROXY_RADIUS_SCALE = 3.0

SOMA_PROXY_MIN_RADIUS = 500.0
SOMA_PROXY_MAX_RADIUS = 20000.0

# Set None to disable nearest-contact filtering.
SOMA_PROXY_MAX_NEAREST_CONTACT_DISTANCE = 5000.0

SOMA_PROXY_MAX_CONTACT_POINTS_FOR_FIT = 100000
SOMA_PROXY_RANDOM_SEED = 0

SYNAPSE_BATCH_SIZE = 400

# ----------------------------
# Historical morphology outputs
# ----------------------------
# These are from the previous soma-in-Raphe-superior clustering run.
# They are examples/reference results only; they do not define the fresh query.
HISTORICAL_ASSIGNMENTS_CSV = Path(
    "data/raphe_superior_soma_in_roi_active_assignments.csv"
)

HISTORICAL_UMAP_CSV = Path(
    "data/raphe_superior_soma_in_roi_active_umap.csv"
)

HISTORICAL_CLUSTER_ID_TO_SHOW = 7

# Optional shell used only for nicer soma plots.
BRAIN_SHELL_NPZ = Path("data/fish2_clean_brain_shell.npz")

# ----------------------------
# Output/cache paths
# ----------------------------
QUERY_OUTPUT_ROOT = Path("outputs/student_roi_queries")
QUERY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path("student_cache")
SKEL_DIR = CACHE_DIR / "skeletons"
SKEL_DIR.mkdir(parents=True, exist_ok=True)

MAX_FRESH_SKELETONS_TO_PLOT = 120
MAX_HISTORICAL_CLUSTER_SKELETONS_TO_PLOT = None

In [ ]:
# ============================================================
# 3. Connect to Fish2 neuPrint
# ============================================================
token = str(NEUPRINT_TOKEN).strip()

if (
    not token
    or token == "PASTE_NEUPRINT_TOKEN_HERE"
    or "PASTE_NEUPRINT_TOKEN_HERE" in token
):
    raise RuntimeError(
        "Paste a valid neuPrint token into NEUPRINT_TOKEN "
        "in the configuration cell above."
    )

c = Client(
    NEUPRINT_SERVER,
    dataset=NEUPRINT_DATASET,
    token=token,
)

print("Connected:", c.server, "| dataset:", c.dataset)

In [ ]:
# ============================================================
# 4. List every ROI and select matching ROI names
# ============================================================
all_rois = sorted(c.meta["roiInfo"].keys())

roi_table = pd.DataFrame({"roi": all_rois})
roi_table["matches_query"] = roi_table["roi"].str.contains(
    ROI_QUERY,
    case=False,
    regex=False,
)

print(f"Total ROI names in Fish2 neuPrint: {len(all_rois):,}")
display(roi_table)

if ROI_MATCH_MODE != "contains":
    raise ValueError("This notebook currently supports ROI_MATCH_MODE='contains'.")

selected_rois = [
    roi
    for roi in all_rois
    if ROI_QUERY.lower() in roi.lower()
]

print("\nSelected ROI names:")
for roi in selected_rois:
    print("  ", roi)

if not selected_rois:
    raise RuntimeError(
        f"No ROI names matched ROI_QUERY={ROI_QUERY!r}. "
        "Change ROI_QUERY and rerun."
    )

In [ ]:
# ============================================================
# 5. FRESH neuPrint query: all neurons touching selected ROI(s)
# ============================================================
touching_neurons, touching_roi_info = fetch_neurons(
    NC(rois=selected_rois, roi_req="any"),
    client=c,
)

touching_neurons = (
    touching_neurons
    .copy()
    .drop_duplicates("bodyId")
    .sort_values("bodyId")
    .reset_index(drop=True)
)

touching_neurons["bodyId"] = touching_neurons["bodyId"].astype(int)

if MAX_TOUCHING_NEURONS is not None:
    touching_neurons = touching_neurons.iloc[
        :int(MAX_TOUCHING_NEURONS)
    ].copy()

touching_ids = touching_neurons["bodyId"].astype(int).tolist()

query_slug = re.sub(
    r"[^A-Za-z0-9._-]+",
    "_",
    ROI_QUERY.strip(),
).strip("_") or "roi_query"

query_out_dir = QUERY_OUTPUT_ROOT / query_slug
query_out_dir.mkdir(parents=True, exist_ok=True)

all_touching_csv = query_out_dir / f"{query_slug}_all_touching_neurons.csv"
all_touching_ids_csv = query_out_dir / f"{query_slug}_all_touching_bodyIds.csv"

touching_neurons.to_csv(all_touching_csv, index=False)
pd.DataFrame({"bodyId": touching_ids}).to_csv(
    all_touching_ids_csv,
    index=False,
)

print(f"Fresh touching-neuron count: {len(touching_ids):,}")
print("Saved:", all_touching_csv)
print("Saved:", all_touching_ids_csv)

display(touching_neurons.head(20))

print("\nComplete fresh touching body-ID list:")
print(touching_ids)

In [ ]:
# ============================================================
# 6. Extract soma/root coordinates from the fresh neuron table
# ============================================================
def location_to_xyz(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, dict):
        if {"x", "y", "z"}.issubset(value):
            return np.array(
                [value["x"], value["y"], value["z"]],
                dtype=float,
            )
        if "coordinates" in value and len(value["coordinates"]) >= 3:
            return np.array(value["coordinates"][:3], dtype=float)

    for attrs in [("x", "y", "z"), ("X", "Y", "Z")]:
        if all(hasattr(value, a) for a in attrs):
            return np.array(
                [getattr(value, a) for a in attrs],
                dtype=float,
            )

    if isinstance(value, (list, tuple, np.ndarray)) and len(value) >= 3:
        return np.array(value[:3], dtype=float)

    # String fallback.
    nums = re.findall(
        r"[-+]?\d*\.\d+|[-+]?\d+",
        str(value),
    )
    if len(nums) >= 3:
        return np.array(
            [float(n) for n in nums[:3]],
            dtype=float,
        )

    return None


def add_soma_xyz_columns(neurons):
    neurons = neurons.copy()

    location_col = None
    for candidate in [
        "somaLocation",
        "tosomaLocation",
        "rootLocation",
        "location",
    ]:
        if candidate in neurons.columns:
            location_col = candidate
            break

    if location_col is None:
        raise RuntimeError(
            "No soma/root location column found. "
            f"Columns: {list(neurons.columns)}"
        )

    xyz = neurons[location_col].apply(location_to_xyz)

    neurons["soma_x"] = [
        float(v[0]) if v is not None and np.isfinite(v).all() else np.nan
        for v in xyz
    ]
    neurons["soma_y"] = [
        float(v[1]) if v is not None and np.isfinite(v).all() else np.nan
        for v in xyz
    ]
    neurons["soma_z"] = [
        float(v[2]) if v is not None and np.isfinite(v).all() else np.nan
        for v in xyz
    ]

    n_valid = neurons[
        ["soma_x", "soma_y", "soma_z"]
    ].notna().all(axis=1).sum()

    print(
        f"Using {location_col} as soma/root coordinate column. "
        f"Coordinates available for {n_valid:,} / {len(neurons):,} cells."
    )

    return neurons, location_col


touching_neurons_xyz, soma_location_column = add_soma_xyz_columns(
    touching_neurons
)

display(
    touching_neurons_xyz[
        ["bodyId", "soma_x", "soma_y", "soma_z"]
    ].head(20)
)

In [ ]:
# ============================================================
# 7. First choice: explicit soma-ROI metadata, if available
# ============================================================
def explicit_soma_roi_filter(neurons, rois):
    roi_cols = [
        col
        for col in neurons.columns
        if "soma" in col.lower() and "roi" in col.lower()
    ]

    if not roi_cols:
        return None, None

    roi_set = set(rois)

    for col in roi_cols:
        keep = neurons[col].apply(
            lambda value: (
                any(roi in str(value) for roi in roi_set)
                if pd.notna(value)
                else False
            )
        )

        if keep.any():
            out = neurons.loc[keep].copy()
            out["soma_in_roi_filter_source"] = (
                f"explicit neuPrint soma ROI column: {col}"
            )
            return out, col

    return None, roi_cols


explicit_soma_in_roi, explicit_roi_column = explicit_soma_roi_filter(
    touching_neurons_xyz,
    selected_rois,
)

if explicit_soma_in_roi is not None:
    print(
        "Fresh explicit soma-ROI metadata filter succeeded."
    )
    print("Column:", explicit_roi_column)
    print(
        "Selected cells:",
        len(explicit_soma_in_roi),
        "/",
        len(touching_neurons_xyz),
    )
else:
    print(
        "No usable explicit soma-ROI metadata match was found. "
        "The next cell will run the live spatial proxy."
    )

In [ ]:
# ============================================================
# 8. Live fallback: build selected-ROI synapse/contact cloud
# ============================================================
def make_synapse_criteria_for_rois(rois):
    # Handles non-primary ROIs such as Raphe_Superior.
    try:
        return SC(rois=rois, primary_only=False)
    except TypeError:
        return SC(rois=rois, include_nonprimary=True)


def fetch_roi_synapse_cloud(body_ids, rois, batch_size=400):
    body_ids = [int(x) for x in body_ids]
    sc = make_synapse_criteria_for_rois(rois)

    batches = []

    for start in tqdm(
        range(0, len(body_ids), batch_size),
        desc="Fetching live in-ROI synapses",
    ):
        batch = body_ids[start:start + batch_size]

        try:
            syn = fetch_synapses(
                NC(bodyId=batch),
                sc,
                client=c,
            )
        except Exception as exc:
            print(
                f"Batch {start}:{start+len(batch)} failed:",
                repr(exc),
            )
            continue

        if syn is not None and len(syn):
            batches.append(syn.copy())

    if not batches:
        raise RuntimeError(
            "No in-ROI synapse/contact points were returned."
        )

    syn = pd.concat(batches, ignore_index=True)

    if {"x", "y", "z"}.issubset(syn.columns):
        pass
    elif "location" in syn.columns:
        xyz = syn["location"].apply(location_to_xyz)
        syn["x"] = [v[0] if v is not None else np.nan for v in xyz]
        syn["y"] = [v[1] if v is not None else np.nan for v in xyz]
        syn["z"] = [v[2] if v is not None else np.nan for v in xyz]
    else:
        raise RuntimeError(
            "Could not find x/y/z or location columns in synapse table. "
            f"Columns: {list(syn.columns)}"
        )

    syn = syn.dropna(subset=["x", "y", "z"]).copy()

    for col in ["x", "y", "z"]:
        syn[col] = syn[col].astype(float)

    return syn


if explicit_soma_in_roi is None:
    roi_synapse_cloud = fetch_roi_synapse_cloud(
        touching_ids,
        selected_rois,
        batch_size=SYNAPSE_BATCH_SIZE,
    )

    roi_synapse_cloud_csv = (
        query_out_dir /
        f"{query_slug}_fresh_in_roi_synapse_contact_points.csv"
    )
    roi_synapse_cloud.to_csv(
        roi_synapse_cloud_csv,
        index=False,
    )

    print(
        f"Fresh in-ROI synapse/contact points: "
        f"{len(roi_synapse_cloud):,}"
    )
    print("Saved:", roi_synapse_cloud_csv)

else:
    roi_synapse_cloud = None
    print(
        "Skipping spatial proxy cloud because explicit soma-ROI "
        "metadata already succeeded."
    )

In [ ]:
# ============================================================
# 9. FRESH soma-in-ROI filtering + CSV/list export
# ============================================================
def filter_somas_by_live_contact_cloud(neurons, synapse_cloud):
    neurons = neurons.copy()

    coords = neurons[
        ["soma_x", "soma_y", "soma_z"]
    ].to_numpy(float)

    valid_soma = np.isfinite(coords).all(axis=1)

    if valid_soma.sum() == 0:
        raise RuntimeError(
            "No valid soma/root coordinates available."
        )

    pts = synapse_cloud[
        ["x", "y", "z"]
    ].to_numpy(float)

    pts = pts[np.isfinite(pts).all(axis=1)]

    if len(pts) == 0:
        raise RuntimeError(
            "No valid in-ROI contact points available."
        )

    if len(pts) > SOMA_PROXY_MAX_CONTACT_POINTS_FOR_FIT:
        rng = np.random.default_rng(SOMA_PROXY_RANDOM_SEED)
        choose = rng.choice(
            len(pts),
            SOMA_PROXY_MAX_CONTACT_POINTS_FOR_FIT,
            replace=False,
        )
        pts_fit = pts[choose]
    else:
        pts_fit = pts

    # Conservative percentile bounding box.
    lo = (
        np.percentile(
            pts_fit,
            SOMA_PROXY_PERCENTILE_LOW,
            axis=0,
        )
        - SOMA_PROXY_PADDING
    )

    hi = (
        np.percentile(
            pts_fit,
            SOMA_PROXY_PERCENTILE_HIGH,
            axis=0,
        )
        + SOMA_PROXY_PADDING
    )

    in_bbox = (
        valid_soma
        & np.all(coords >= lo[None, :], axis=1)
        & np.all(coords <= hi[None, :], axis=1)
    )

    # PCA ellipsoid around the live in-ROI point cloud.
    center = np.median(pts_fit, axis=0)

    keep = in_bbox.copy()
    radii = None
    eigvecs = None

    if SOMA_PROXY_USE_ELLIPSOID:
        cov = np.cov(
            (pts_fit - center[None, :]).T
        )

        eigvals, eigvecs = np.linalg.eigh(cov)

        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]

        radii = (
            np.sqrt(np.maximum(eigvals, 1e-9))
            * SOMA_PROXY_RADIUS_SCALE
        )

        radii = np.clip(
            radii,
            SOMA_PROXY_MIN_RADIUS,
            SOMA_PROXY_MAX_RADIUS,
        )

        centered = coords - center[None, :]
        uvw = centered @ eigvecs

        in_ellipsoid = (
            np.sum(
                (uvw / radii[None, :]) ** 2,
                axis=1,
            )
            <= 1.0
        )

        # Same logic as the local workflow:
        # union ellipsoid with conservative bbox.
        keep = (
            valid_soma
            & (in_ellipsoid | in_bbox)
        )

    nearest_dist = np.full(
        len(neurons),
        np.nan,
        dtype=float,
    )

    if SOMA_PROXY_MAX_NEAREST_CONTACT_DISTANCE is not None:
        tree = cKDTree(pts_fit)

        d, _ = tree.query(
            coords[valid_soma],
            k=1,
        )

        nearest_dist[valid_soma] = d

        keep = (
            keep
            & (
                nearest_dist
                <= float(
                    SOMA_PROXY_MAX_NEAREST_CONTACT_DISTANCE
                )
            )
        )

    out = neurons.loc[keep].copy()

    out["soma_in_roi_filter_source"] = (
        "fresh live synapse/contact cloud spatial proxy"
    )

    out["soma_proxy_nearest_contact_distance"] = (
        nearest_dist[keep]
    )

    out["soma_proxy_in_bbox"] = (
        in_bbox[keep]
    )

    geometry = {
        "bbox_low": lo,
        "bbox_high": hi,
        "center": center,
        "radii": (
            radii
            if radii is not None
            else np.array([np.nan, np.nan, np.nan])
        ),
        "eigvecs": (
            eigvecs
            if eigvecs is not None
            else np.eye(3)
        ),
    }

    return out, geometry


if explicit_soma_in_roi is not None:
    fresh_soma_in_roi = explicit_soma_in_roi.copy()
    fresh_filter_method = "explicit_soma_roi_metadata"
    proxy_geometry = None

else:
    fresh_soma_in_roi, proxy_geometry = (
        filter_somas_by_live_contact_cloud(
            touching_neurons_xyz,
            roi_synapse_cloud,
        )
    )
    fresh_filter_method = "live_synapse_contact_cloud_proxy"


fresh_soma_in_roi = (
    fresh_soma_in_roi
    .drop_duplicates("bodyId")
    .sort_values("bodyId")
    .reset_index(drop=True)
)

fresh_soma_in_roi_ids = (
    fresh_soma_in_roi["bodyId"]
    .astype(int)
    .tolist()
)

fresh_full_csv = (
    query_out_dir /
    f"{query_slug}_fresh_soma_in_roi_neurons.csv"
)

fresh_ids_csv = (
    query_out_dir /
    f"{query_slug}_fresh_soma_in_roi_bodyIds.csv"
)

fresh_soma_in_roi.to_csv(
    fresh_full_csv,
    index=False,
)

pd.DataFrame(
    {"bodyId": fresh_soma_in_roi_ids}
).to_csv(
    fresh_ids_csv,
    index=False,
)

if proxy_geometry is not None:
    proxy_npz = (
        query_out_dir /
        f"{query_slug}_fresh_soma_proxy_geometry.npz"
    )
    np.savez_compressed(
        proxy_npz,
        **proxy_geometry,
    )
    print("Saved:", proxy_npz)


print("\nFRESH soma-in-ROI result")
print("------------------------")
print("ROI query:", ROI_QUERY)
print("Matched ROIs:", selected_rois)
print("Filter method:", fresh_filter_method)
print("All touching neurons:", len(touching_ids))
print(
    "Cells with soma/root coordinates:",
    touching_neurons_xyz[
        ["soma_x", "soma_y", "soma_z"]
    ].notna().all(axis=1).sum(),
)
print(
    "Fresh soma-in-ROI cells:",
    len(fresh_soma_in_roi_ids),
)

print("\nSaved:", fresh_full_csv)
print("Saved:", fresh_ids_csv)

display(fresh_soma_in_roi.head(20))

print("\nComplete fresh soma-in-ROI body-ID list:")
print(fresh_soma_in_roi_ids)

In [ ]:
# ============================================================
# 10. Plot FRESH selected soma positions in 3D
# ============================================================
fig = go.Figure()

if BRAIN_SHELL_NPZ.exists():
    shell = np.load(BRAIN_SHELL_NPZ)

    vertices = np.asarray(
        shell["vertices"],
        dtype=float,
    )
    faces = np.asarray(
        shell["faces"],
        dtype=int,
    )

    fig.add_trace(
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color="rgb(185,190,195)",
            opacity=0.05,
            flatshading=True,
            name="Brain shell",
            showscale=False,
        )
    )

fig.add_trace(
    go.Scatter3d(
        x=fresh_soma_in_roi["soma_x"],
        y=fresh_soma_in_roi["soma_y"],
        z=fresh_soma_in_roi["soma_z"],
        mode="markers",
        marker=dict(
            size=4,
            color="gold",
            opacity=0.9,
        ),
        text=(
            fresh_soma_in_roi["bodyId"]
            .astype(str)
        ),
        hovertemplate=(
            "bodyId=%{text}"
            "<extra></extra>"
        ),
        name=(
            f"Fresh soma-in-ROI "
            f"(n={len(fresh_soma_in_roi)})"
        ),
    )
)

fig.update_layout(
    title=(
        f"Fresh soma-in-ROI selection: {ROI_QUERY}"
        f"<br><sup>{fresh_filter_method}</sup>"
    ),
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        bgcolor="white",
    ),
    paper_bgcolor="white",
    height=800,
    margin=dict(l=0, r=0, t=75, b=0),
)

fig.show()

In [ ]:
# ============================================================
# 11. Skeleton helpers
# ============================================================
def fetch_or_load_skeleton(body_id, heal=True):
    body_id = int(body_id)

    path = SKEL_DIR / f"{body_id}.swc"

    if not path.exists():
        try:
            c.fetch_skeleton(
                body_id,
                heal=heal,
                export_path=str(path),
            )
        except Exception as exc:
            print(
                f"Could not fetch skeleton for {body_id}:",
                repr(exc),
            )
            return None

    rows = []

    with open(path, "r", errors="ignore") as handle:
        for line in handle:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            parts = line.split()

            if len(parts) < 7:
                continue

            try:
                rows.append([
                    int(float(parts[0])),
                    int(float(parts[1])),
                    float(parts[2]),
                    float(parts[3]),
                    float(parts[4]),
                    float(parts[5]),
                    int(float(parts[6])),
                ])
            except ValueError:
                continue

    if not rows:
        return None

    return pd.DataFrame(
        rows,
        columns=[
            "node_id",
            "type",
            "x",
            "y",
            "z",
            "radius",
            "parent_id",
        ],
    )


def swc_edges(swc):
    xyz = {
        int(row.node_id): (
            float(row.x),
            float(row.y),
            float(row.z),
        )
        for row in swc.itertuples()
    }

    xs, ys, zs = [], [], []

    for row in swc.itertuples():
        node_id = int(row.node_id)
        parent_id = int(row.parent_id)

        if parent_id < 0 or parent_id not in xyz:
            continue

        x1, y1, z1 = xyz[parent_id]
        x2, y2, z2 = xyz[node_id]

        xs.extend([x1, x2, None])
        ys.extend([y1, y2, None])
        zs.extend([z1, z2, None])

    return xs, ys, zs


def plot_skeleton_set(
    body_ids,
    title,
    color,
    max_neurons=None,
    width=1.4,
):
    ids = [int(x) for x in body_ids]

    if max_neurons is not None:
        ids = ids[:int(max_neurons)]

    fig = go.Figure()
    plotted = []

    for body_id in tqdm(
        ids,
        desc="Loading skeletons",
    ):
        swc = fetch_or_load_skeleton(body_id)

        if swc is None:
            continue

        xs, ys, zs = swc_edges(swc)

        fig.add_trace(
            go.Scatter3d(
                x=xs,
                y=ys,
                z=zs,
                mode="lines",
                line=dict(
                    color=color,
                    width=width,
                ),
                name=str(body_id),
                hoverinfo="name",
                showlegend=False,
            )
        )

        plotted.append(body_id)

    fig.update_layout(
        title=(
            f"{title}"
            f"<br><sup>n={len(plotted)}</sup>"
        ),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data",
            bgcolor="white",
        ),
        paper_bgcolor="white",
        height=800,
        margin=dict(l=0, r=0, t=70, b=0),
    )

    fig.show()

    return fig, plotted

In [ ]:
# ============================================================
# 12. Plot FRESH soma-in-ROI skeletons
# ============================================================
fresh_soma_skeleton_fig, fresh_soma_skeleton_ids = (
    plot_skeleton_set(
        fresh_soma_in_roi_ids,
        title=(
            f"Fresh soma-in-ROI neurons for {ROI_QUERY}"
        ),
        color="rgb(70,130,220)",
        max_neurons=MAX_FRESH_SKELETONS_TO_PLOT,
        width=1.2,
    )
)

# Historical morphology example: previously identified cluster 7

The following cells do **not** define the fresh ROI query.

They load the saved clustering outputs from the previous local soma-in-Raphe-superior NBLAST analysis so students can inspect the previously identified cluster 7.

In [ ]:
# ============================================================
# 13. Load previous morphology assignments and access cluster 7
# ============================================================
if not HISTORICAL_ASSIGNMENTS_CSV.exists():
    raise FileNotFoundError(
        f"Missing historical assignment file:\n"
        f"  {HISTORICAL_ASSIGNMENTS_CSV}\n\n"
        "Copy the active assignment CSV from the previous "
        "soma-in-Raphe-superior clustering run into data/ "
        "using that exact filename."
    )

historical_assignments = pd.read_csv(
    HISTORICAL_ASSIGNMENTS_CSV
)

required_cols = {"bodyId", "cluster"}

missing = (
    required_cols
    - set(historical_assignments.columns)
)

if missing:
    raise RuntimeError(
        f"Historical assignment CSV is missing: "
        f"{sorted(missing)}"
    )

historical_assignments = (
    historical_assignments.copy()
)

historical_assignments["bodyId"] = (
    historical_assignments["bodyId"].astype(int)
)

historical_assignments["cluster"] = (
    historical_assignments["cluster"].astype(int)
)

historical_cluster7 = (
    historical_assignments.loc[
        historical_assignments["cluster"]
        == int(HISTORICAL_CLUSTER_ID_TO_SHOW)
    ]
    .copy()
    .sort_values("bodyId")
    .reset_index(drop=True)
)

historical_cluster7_ids = (
    historical_cluster7["bodyId"]
    .astype(int)
    .tolist()
)

print(
    "Available historical clusters:",
    sorted(
        historical_assignments[
            "cluster"
        ].unique().tolist()
    ),
)

print(
    f"Historical cluster "
    f"{HISTORICAL_CLUSTER_ID_TO_SHOW}: "
    f"n={len(historical_cluster7_ids)}"
)

display(historical_cluster7)

print("\nHistorical cluster 7 body IDs:")
print(historical_cluster7_ids)

In [ ]:
# ============================================================
# 14. Plot historical cluster 7 skeletons
# ============================================================
historical_cluster7_fig, historical_cluster7_plotted_ids = (
    plot_skeleton_set(
        historical_cluster7_ids,
        title=(
            "Previous soma-in-Raphe-superior "
            f"morphology cluster "
            f"{HISTORICAL_CLUSTER_ID_TO_SHOW}"
        ),
        color="rgb(235,140,35)",
        max_neurons=MAX_HISTORICAL_CLUSTER_SKELETONS_TO_PLOT,
        width=2.0,
    )
)

In [ ]:
# ============================================================
# 15. Optional: show historical cluster 7 on historical UMAP
# ============================================================
if HISTORICAL_UMAP_CSV.exists():
    historical_umap = pd.read_csv(
        HISTORICAL_UMAP_CSV
    )

    historical_umap["bodyId"] = (
        historical_umap["bodyId"].astype(int)
    )

    if {"umap1", "umap2"}.issubset(
        historical_umap.columns
    ):
        xcol, ycol = "umap1", "umap2"

    elif {"UMAP1", "UMAP2"}.issubset(
        historical_umap.columns
    ):
        xcol, ycol = "UMAP1", "UMAP2"

    else:
        raise RuntimeError(
            "Historical UMAP CSV needs "
            "umap1/umap2 or UMAP1/UMAP2 columns. "
            f"Found: {list(historical_umap.columns)}"
        )

    plot_df = historical_umap.merge(
        historical_assignments[
            ["bodyId", "cluster"]
        ],
        on="bodyId",
        how="left",
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=plot_df[xcol],
            y=plot_df[ycol],
            mode="markers",
            marker=dict(
                size=7,
                color="lightgray",
                opacity=0.65,
            ),
            text=plot_df["bodyId"].astype(str),
            hovertemplate=(
                "bodyId=%{text}"
                "<extra></extra>"
            ),
            name="All historical embedded cells",
        )
    )

    hit = plot_df.loc[
        plot_df["bodyId"].isin(
            historical_cluster7_ids
        )
    ]

    fig.add_trace(
        go.Scatter(
            x=hit[xcol],
            y=hit[ycol],
            mode="markers",
            marker=dict(
                size=11,
                color="orange",
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            text=hit["bodyId"].astype(str),
            hovertemplate=(
                "bodyId=%{text}"
                "<extra></extra>"
            ),
            name=(
                f"Historical cluster "
                f"{HISTORICAL_CLUSTER_ID_TO_SHOW}"
            ),
        )
    )

    fig.update_layout(
        title=(
            f"Historical cluster "
            f"{HISTORICAL_CLUSTER_ID_TO_SHOW} "
            "on saved UMAP"
        ),
        xaxis_title=xcol,
        yaxis_title=ycol,
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=700,
    )

    fig.show()

else:
    print(
        "Optional historical UMAP file not found:",
        HISTORICAL_UMAP_CSV,
    )

## Files needed from the previous local clustering run

Only the **historical morphology example** requires committed local-output files:

```text
data/
├── raphe_superior_soma_in_roi_active_assignments.csv
├── raphe_superior_soma_in_roi_active_umap.csv
└── fish2_clean_brain_shell.npz
```

The fresh ROI query itself does **not** use a precomputed soma-membership CSV.

Every fresh run writes its own CSV outputs under:

```text
outputs/student_roi_queries/<ROI_QUERY>/
```

including:

```text
<query>_all_touching_neurons.csv
<query>_all_touching_bodyIds.csv
<query>_fresh_soma_in_roi_neurons.csv
<query>_fresh_soma_in_roi_bodyIds.csv
```